# Geospatial Analysis using DuckDB database.
It reads Heoparquet files using python and creates a spatial map of cities in Asia.


In [ ]:
# Core analytics
!pip install -U duckdb

# Spatial + visualization stack
!pip install geopandas pyarrow shapely folium matplotlib

In [ ]:
import duckdb
import geopandas as gpd
import pandas as pd

import folium
import matplotlib.pyplot as plt


In [ ]:
con = duckdb.connect()

con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

In [ ]:
cities = "read_parquet('https://storage.googleapis.com/foss4g_workshop/geonames_citiea.geo.parquet')"

states = "read_parquet('https://storage.googleapis.com/foss4g_workshop/asia_states_geoboundaries.geoparquet')"


Let's count cities per states.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE state_city_count AS
SELECT
    s.country_name,
    s.state_name,
    COUNT(c.*) AS city_count,
    s.geometry
FROM {states} s
LEFT JOIN {cities} c
ON ST_Within(c.geom, s.geometry)
GROUP BY
    s.country_name,
    s.state_name,
    s.geometry
""")


Let's bring the states table into geopandas.

In [ ]:
import geopandas as gpd

df_states = con.execute("""
SELECT
    country_name,
    state_name,
    city_count,
    ST_AsText(geometry) AS wkt
FROM state_city_count
""").df()

gdf_states = gpd.GeoDataFrame(
    df_states,
    geometry=gpd.GeoSeries.from_wkt(df_states["wkt"]),
    crs="EPSG:4326"
).drop(columns="wkt")


Let's bring the cities table into geopandas.

In [ ]:
df_cities = con.execute(f"""
SELECT
    name,
    population,
    avg_tmax,
    ST_AsText(geom) AS wkt
FROM {cities}
""").df()

gdf_cities = gpd.GeoDataFrame(
    df_cities,
    geometry=gpd.GeoSeries.from_wkt(df_cities["wkt"]),
    crs="EPSG:4326"
).drop(columns="wkt")


In [ ]:
import numpy as np

gdf_cities["marker_size"] = np.sqrt(gdf_cities["population"]) / 40



In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 9))

# --- States: blue shades (urban structure) ---
gdf_states.plot(
    ax=ax,
    column="city_count",
    cmap="Blues",
    linewidth=0.4,
    edgecolor="#444444",
    legend=True,
    legend_kwds={
        "label": "Number of cities (>1M population)",
        "shrink": 0.6
    }
)

# --- Cities: red points, size = population ---
gdf_cities.plot(
    ax=ax,
    color="#cc0000",
    markersize=gdf_cities["marker_size"],
    alpha=0.65,
    edgecolor="white",
    linewidth=0.3
)

ax.set_title(
    "Urban Structure (States) and Large Cities (Population-scaled)",
    fontsize=15
)
ax.axis("off")
